In [1]:
import os

In [2]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main\\research'

In [3]:
from pathlib import Path
import os

_cwd = Path.cwd().resolve()
if _cwd.name == "research":
    os.chdir(_cwd.parent)

In [4]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main'

In [5]:
import os

os.environ.setdefault("KERAS_BACKEND", "torch")
try:
    import tensorflow as tf
except ModuleNotFoundError:
    import keras as _keras

    class _TF:
        keras = _keras

    tf = _TF()

In [6]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [8]:
import sys
import os
import unittest

if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

sys.path.insert(0, os.path.abspath("src"))

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chicken-fecal-images",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [10]:
from urllib.parse import urlparse

In [11]:
from keras_preprocessing.image import ImageDataGenerator


class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=str(self.config.training_data),
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(str(path))
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        steps = (
            self.valid_generator.samples + self.valid_generator.batch_size - 1
        ) // self.valid_generator.batch_size

        def _val_batches():
            while True:
                yield self.valid_generator.next()

        self.score = self.model.evaluate(_val_batches(), steps=steps)

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    

In [12]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
   raise e

Found 116 images belonging to 2 classes.


1/8 ━━━━━━━━━━━━━━━━━━━━ 23s 3s/step - accuracy: 1.0000 - loss: 1.1921e-07

2/8 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step - accuracy: 1.0000 - loss: 1.1921e-07

3/8 ━━━━━━━━━━━━━━━━━━━━ 18s 4s/step - accuracy: 1.0000 - loss: 1.1921e-07

4/8 ━━━━━━━━━━━━━━━━━━━━ 14s 4s/step - accuracy: 0.9766 - loss: 0.3778    

5/8 ━━━━━━━━━━━━━━━━━━━━ 11s 4s/step - accuracy: 0.9263 - loss: 1.1887

6/8 ━━━━━━━━━━━━━━━━━━━━ 7s 4s/step - accuracy: 0.8726 - loss: 2.0539 

7/8 ━━━━━━━━━━━━━━━━━━━━ 3s 4s/step - accuracy: 0.8219 - loss: 2.8707

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7817 - loss: 3.5192

8/8 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - accuracy: 0.5000 - loss: 8.0590
